# 🤖 Coding Agent Avanzado — Kotlin / Spring Boot

**TP Final - Inteligencia Artificial**

Sistema multi-agente para análisis, corrección y mejora de proyectos Kotlin/Spring Boot.

### Arquitectura
- **Agente Principal (Orquestador)**: Recibe tareas, coordina subagentes, genera reportes
- **Explorer**: Analiza estructura del repositorio, arquitectura, dependencias
- **Researcher**: Busca en RAG (Kotlin/Spring Boot docs) y web
- **Implementer**: Propone y realiza cambios de código
- **Tester**: Ejecuta tests con Gradle, reporta resultados
- **Reviewer**: Revisa cambios y valida calidad

### Características
- 🔍 RAG sobre documentación Kotlin/Spring Boot (ChromaDB + OpenAI Embeddings)
- 💾 Memoria persistente por proyecto (JSON)
- 🔒 Políticas de seguridad configurables (YAML)
- 📊 Observabilidad con Langfuse
- 🔄 Detección de loops y gestión de contexto
- 🔌 Sistema de registro de herramientas (plugin-style)


### 📦 Instalación de dependencias

In [ ]:
# =============================================================================
# Cell 01 — Setup: Dependencies, JDK, Gradle
# =============================================================================
# Installs all required Python packages, patches sqlite3 for ChromaDB
# compatibility, installs JDK and Gradle 8.7, and configures PATH.
# This is the ONLY cell that uses ! shell commands.
# =============================================================================

# --- Python dependencies ---
!pip install -q openai chromadb langfuse tavily-python pyyaml pysqlite3-binary

# --- SQLite3 override for ChromaDB compatibility in Colab ---
# ChromaDB requires a newer SQLite3 than Colab ships by default.
# pysqlite3-binary provides a compatible version.
__import__('pysqlite3')
import sys
sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

# --- Install JDK (headless) ---
!apt-get update -qq && apt-get install -y -qq default-jdk-headless > /dev/null 2>&1

# --- Install Gradle 8.7 ---
!wget -q https://services.gradle.org/distributions/gradle-8.7-bin.zip && unzip -q -o gradle-8.7-bin.zip -d /opt/ > /dev/null 2>&1

# --- Configure environment variables ---
import os

os.environ['JAVA_HOME'] = '/usr/lib/jvm/default-java'
os.environ['GRADLE_HOME'] = '/opt/gradle-8.7'
os.environ['PATH'] = f"/opt/gradle-8.7/bin:{os.environ['JAVA_HOME']}/bin:{os.environ['PATH']}"

# --- Verification ---
print("=" * 60)
print("✅ Dependencias Python instaladas correctamente")
print(f"✅ JAVA_HOME = {os.environ['JAVA_HOME']}")
print(f"✅ GRADLE_HOME = {os.environ['GRADLE_HOME']}")
print("✅ JDK y Gradle 8.7 instalados y configurados en PATH")
print("✅ SQLite3 parcheado para compatibilidad con ChromaDB")
print("=" * 60)


### ⚙️ Configuración del cliente y políticas del agente

In [ ]:
# =============================================================================
# Cell 02 — Configuration: OpenAI Client, YAML Config, Permission Checker
# =============================================================================
# Sets up the OpenAI client, defines the agent config YAML (workspace, model,
# ecosystem, permissions with deny/require_approval lists), and provides
# load_config() and check_permission() functions.
# =============================================================================

import os
import yaml
import fnmatch
from openai import OpenAI
from google.colab import userdata

# --- OpenAI Client ---
client: OpenAI = OpenAI(api_key=userdata.get("OPENAI_API_KEY"))

# --- Model Configuration ---
MODEL: str = "gpt-5-nano"
EMBEDDING_MODEL: str = "text-embedding-3-small"

# --- Tavily API Key (for web search) ---
TAVILY_API_KEY: str = userdata.get("TAVILY_API_KEY")

# --- Global config and workspace (populated by load_config()) ---
config: dict = {}
WORKSPACE: str = ""

# --- Write the agent configuration YAML to disk ---
_CONFIG_YAML_CONTENT: str = """\
# Agent Configuration — Kotlin/Spring Boot Coding Agent
workspace: "./workspace/demo-spring-app"

model:
  name: "gpt-5-nano"
  temperature: 0.2
  max_tokens: 4096

ecosystem:
  language: "kotlin"
  framework: "spring-boot"
  build_tool: "gradle"
  test_framework: "junit5"

permissions:
  read:
    deny:
      - "/etc/shadow"
      - "/etc/passwd"
      - "**/.env"
      - "**/secrets/**"
      - "**/*.pem"
      - "**/*.key"
  write:
    deny:
      - "/etc/**"
      - "/usr/**"
      - "/bin/**"
      - "/sbin/**"
      - "**/.git/**"
      - "**/node_modules/**"
  command:
    deny:
      - "rm -rf /"
      - "rm -rf /*"
      - "sudo *"
      - "curl * | bash"
      - "wget * | bash"
      - "chmod 777 *"
      - "shutdown*"
      - "reboot*"
    require_approval:
      - "rm *"
      - "mv *"
      - "gradle clean*"
      - "kill *"

max_iterations: 25
"""

_CONFIG_PATH: str = "agent.config.yaml"
if not os.path.exists(_CONFIG_PATH):
    with open(_CONFIG_PATH, "w") as f:
        f.write(_CONFIG_YAML_CONTENT)


def load_config(path: str = "agent.config.yaml") -> dict:
    """
    Load the YAML config file and set global `config` and `WORKSPACE`.

    Args:
        path: Path to the YAML configuration file.

    Returns:
        The parsed configuration dictionary.
    """
    global config, WORKSPACE
    try:
        with open(path, "r") as f:
            config = yaml.safe_load(f)

        # Set the workspace directory (create if needed)
        WORKSPACE = config.get("workspace", "./workspace")
        os.makedirs(WORKSPACE, exist_ok=True)

        return config
    except Exception as e:
        print(f"❌ Error al cargar configuración: {e}")
        config = {}
        WORKSPACE = "./workspace"
        os.makedirs(WORKSPACE, exist_ok=True)
        return config


def check_permission(action_type: str, target: str, config: dict) -> tuple:
    """
    Check if an action is allowed by the security policies.

    Uses fnmatch glob matching against the deny lists in config["permissions"].
    For commands, also checks the require_approval list and prompts the user
    for confirmation via input() if needed.

    Args:
        action_type: One of "read", "write", or "command".
        target: The file path or command string to check.
        config: The loaded configuration dictionary.

    Returns:
        A tuple of (allowed: bool, reason: str).
    """
    try:
        permissions = config.get("permissions", {})
        action_perms = permissions.get(action_type, {})
        deny_list = action_perms.get("deny", [])

        # Check against deny list using glob matching
        for pattern in deny_list:
            if fnmatch.fnmatch(target, pattern):
                return (False, f"Acción denegada: '{target}' coincide con patrón bloqueado '{pattern}'")

        # For commands, also check the require_approval list
        if action_type == "command" and not config.get("supervision_mode"):
            approval_list = action_perms.get("require_approval", [])
            for pattern in approval_list:
                if fnmatch.fnmatch(target, pattern):
                    print(f"\n⚠️  El comando requiere aprobación: {target}")
                    print(f"   (coincide con patrón: '{pattern}')")
                    user_input = input("   ¿Aprobar ejecución? (s/n): ").strip().lower()
                    if user_input not in ("s", "si", "sí", "y", "yes"):
                        return (False, f"Comando rechazado por el usuario: '{target}'")
                    break  # Approved, no need to check further patterns

        return (True, "Permitido")

    except Exception as e:
        # Fail-safe: deny if we can't check permissions
        return (False, f"Error al verificar permisos: {e}")


# --- Initialize config on cell execution ---
load_config()

# --- Print summary ---
print("=" * 60)
print("✅ Cliente OpenAI configurado")
print(f"   Modelo: {MODEL}")
print(f"   Embedding: {EMBEDDING_MODEL}")
print(f"✅ TAVILY_API_KEY configurada")
print(f"✅ Configuración cargada desde: {_CONFIG_PATH}")
print(f"   Workspace: {WORKSPACE}")
print(f"   Ecosistema: {config.get('ecosystem', {}).get('language', '?')}"
      f"/{config.get('ecosystem', {}).get('framework', '?')}")
print(f"   Max iteraciones: {config.get('max_iterations', '?')}")
print(f"   Patrones denegados (read): {len(config.get('permissions', {}).get('read', {}).get('deny', []))}")
print(f"   Patrones denegados (write): {len(config.get('permissions', {}).get('write', {}).get('deny', []))}")
print(f"   Patrones denegados (command): {len(config.get('permissions', {}).get('command', {}).get('deny', []))}")
print(f"   Requieren aprobación: {len(config.get('permissions', {}).get('command', {}).get('require_approval', []))}")
print("=" * 60)


### 📊 Observabilidad con Langfuse

In [ ]:
# =============================================================================
# Cell 03 — Observability: Langfuse Tracer
# =============================================================================
# Provides the Tracer class that logs LLM generations, tool calls, and RAG
# retrievals to Langfuse for monitoring and debugging. Gracefully degrades
# if Langfuse credentials are not configured.
# =============================================================================

import time
from langfuse import Langfuse
from google.colab import userdata

# --- Attempt to load Langfuse credentials ---
_LANGFUSE_PUBLIC_KEY: str = ""
_LANGFUSE_SECRET_KEY: str = ""
_LANGFUSE_HOST: str = "https://cloud.langfuse.com"

try:
    _LANGFUSE_PUBLIC_KEY = userdata.get("LANGFUSE_PUBLIC_KEY")
    _LANGFUSE_SECRET_KEY = userdata.get("LANGFUSE_SECRET_KEY")
except Exception:
    # Keys not configured — Tracer will run in disabled mode
    pass

# Optionally override host
try:
    _host = userdata.get("LANGFUSE_HOST")
    if _host:
        _LANGFUSE_HOST = _host
except Exception:
    pass


class Tracer:
    """
    Observability wrapper around Langfuse for tracing LLM calls,
    tool executions, and RAG retrievals.

    If Langfuse credentials are not available, all methods become
    no-ops to avoid crashing the notebook.
    """

    def __init__(self):
        """Initialize the Langfuse client if credentials are available."""
        self.enabled: bool = False
        self.client = None

        if _LANGFUSE_PUBLIC_KEY and _LANGFUSE_SECRET_KEY:
            try:
                self.client = Langfuse(
                    public_key=_LANGFUSE_PUBLIC_KEY,
                    secret_key=_LANGFUSE_SECRET_KEY,
                    host=_LANGFUSE_HOST,
                )
                self.enabled = True
            except Exception as e:
                print(f"⚠️  No se pudo inicializar Langfuse: {e}")
                self.enabled = False

    def start_trace(self, name: str, user_id: str = "user"):
        """
        Start a new Langfuse trace.

        Args:
            name: Name/identifier for this trace (e.g., the task description).
            user_id: User identifier for grouping traces.

        Returns:
            A Langfuse trace object, or None if disabled.
        """
        if not self.enabled:
            return None
        try:
            trace = self.client.trace(
                name=name,
                user_id=user_id,
            )
            return trace
        except Exception as e:
            print(f"⚠️  Error al crear trace: {e}")
            return None

    def log_llm_call(
        self,
        trace,
        agent_name: str,
        messages: list,
        response,
        duration_ms: float,
    ):
        """
        Log an LLM generation to Langfuse.

        Records the model used, input/output messages, token usage,
        estimated cost, and latency.

        Args:
            trace: The parent Langfuse trace object.
            agent_name: Name of the agent making the call.
            messages: The input messages sent to the LLM.
            response: The OpenAI API response object.
            duration_ms: Wall-clock duration of the call in milliseconds.
        """
        if not self.enabled or trace is None:
            return
        try:
            # Extract token usage from the response
            usage = response.usage if response and hasattr(response, "usage") else None
            prompt_tokens = usage.prompt_tokens if usage else 0
            completion_tokens = usage.completion_tokens if usage else 0
            total_tokens = usage.total_tokens if usage else 0

            # Rough cost estimate (USD) for gpt-5-nano
            # Approximation: $0.10/1M input tokens, $0.40/1M output tokens
            estimated_cost = (prompt_tokens * 0.10 / 1_000_000) + \
                             (completion_tokens * 0.40 / 1_000_000)

            # Extract the output text
            output_text = ""
            if response and response.choices:
                choice = response.choices[0]
                if hasattr(choice.message, "content") and choice.message.content:
                    output_text = choice.message.content

            trace.generation(
                name=f"llm-{agent_name}",
                model=MODEL,
                input=messages,
                output=output_text,
                usage={
                    "promptTokens": prompt_tokens,
                    "completionTokens": completion_tokens,
                    "totalTokens": total_tokens,
                },
                metadata={
                    "agent": agent_name,
                    "duration_ms": round(duration_ms, 2),
                    "estimated_cost_usd": round(estimated_cost, 6),
                },
            )
        except Exception as e:
            print(f"⚠️  Error al registrar llamada LLM: {e}")

    def log_tool_call(
        self,
        trace,
        tool_name: str,
        args: dict,
        result: str,
        duration_ms: float,
    ):
        """
        Log a tool execution as a Langfuse span.

        Args:
            trace: The parent Langfuse trace object.
            tool_name: Name of the tool executed.
            args: Arguments passed to the tool.
            result: The tool's return value (as string).
            duration_ms: Duration of the tool call in milliseconds.
        """
        if not self.enabled or trace is None:
            return
        try:
            trace.span(
                name=f"tool-{tool_name}",
                input=args,
                output=result[:2000] if result else "",  # Truncate long results
                metadata={
                    "tool": tool_name,
                    "duration_ms": round(duration_ms, 2),
                },
            )
        except Exception as e:
            print(f"⚠️  Error al registrar llamada de herramienta: {e}")

    def log_rag_retrieval(self, trace, query: str, results: list):
        """
        Log a RAG retrieval operation as a Langfuse span.

        Args:
            trace: The parent Langfuse trace object.
            query: The search query used.
            results: List of retrieved documents/chunks.
        """
        if not self.enabled or trace is None:
            return
        try:
            trace.span(
                name="rag-retrieval",
                input={"query": query},
                output={"num_results": len(results), "results": results[:10]},
                metadata={
                    "query_length": len(query),
                    "num_results": len(results),
                },
            )
        except Exception as e:
            print(f"⚠️  Error al registrar búsqueda RAG: {e}")

    def flush(self):
        """Flush all pending events to Langfuse."""
        if not self.enabled or self.client is None:
            return
        try:
            self.client.flush()
        except Exception as e:
            print(f"⚠️  Error al hacer flush de Langfuse: {e}")


# --- Global Tracer instance ---
tracer: Tracer = Tracer()

# --- Status message ---
print("=" * 60)
if tracer.enabled:
    print("✅ Langfuse Tracer inicializado correctamente")
    print(f"   Host: {_LANGFUSE_HOST}")
else:
    print("⚠️  Langfuse Tracer en modo deshabilitado (claves no configuradas)")
    print("   El agente funcionará normalmente sin observabilidad.")
print("=" * 60)


### 📋 Estado compartido de tarea (TaskState)

In [ ]:
# =============================================================================
# Cell 04 — State: TaskState
# =============================================================================
# The TaskState class tracks the full lifecycle of a single agent task:
# plan, status, observations, errors, sources consulted, files modified,
# subagent results, and action history (with MD5 hashing for loop detection).
# to_summary() provides an aggressively truncated view for LLM context.
# =============================================================================

import hashlib
import json


class TaskState:
    """
    Mutable state object for a single agent task.

    Tracks everything the orchestrator and subagents need to know about
    the current task: plan steps, progress, observations, errors, sources,
    file modifications, and action history.
    """

    # Valid status transitions
    VALID_STATUSES = (
        "planning", "exploring", "researching", "implementing",
        "testing", "reviewing", "done", "blocked",
    )

    def __init__(self, original_request: str):
        """
        Initialize a new TaskState.

        Args:
            original_request: The user's original task description.
        """
        self.original_request: str = original_request
        self.status: str = "planning"
        self.plan: list = []
        self.current_step: int = 0
        self.subagent_results: dict = {}       # {agent_name: [result_strings]}
        self.sources_consulted: list = []      # [{"type": str, "ref": str, "snippet": str}]
        self.files_modified: list = []
        self.observations: list = []
        self.errors: list = []
        self.iteration_count: int = 0
        self.action_history: list = []         # [{"action": str, "args_hash": str, "result_hash": str}]

    def add_source(self, source_type: str, ref: str, snippet: str):
        """
        Record a consulted source (RAG, web, repo, or memory).

        Args:
            source_type: One of "rag", "web", "repo", "memory".
            ref: Reference identifier (URL, file path, collection name, etc.).
            snippet: A short excerpt from the source.
        """
        self.sources_consulted.append({
            "type": source_type,
            "ref": ref,
            "snippet": snippet[:200],  # Cap snippet length at storage time
        })

    def add_observation(self, text: str):
        """
        Record an observation made during task execution.

        Args:
            text: The observation text.
        """
        self.observations.append(text)

    def mark_file_modified(self, path: str):
        """
        Record that a file was modified.

        Args:
            path: The path to the modified file.
        """
        if path not in self.files_modified:
            self.files_modified.append(path)

    def add_error(self, error: str):
        """
        Record an error encountered during execution.

        Args:
            error: The error message or description.
        """
        self.errors.append(error)

    def record_action(self, action: str, args: str, result: str):
        """
        Record an action for loop detection purposes.

        Uses MD5 hashing on args and result to enable efficient comparison
        without storing full content.

        Args:
            action: The action/tool name.
            args: The arguments (as a string).
            result: The result (as a string).
        """
        args_hash = hashlib.md5(args.encode("utf-8", errors="replace")).hexdigest()
        result_hash = hashlib.md5(result.encode("utf-8", errors="replace")).hexdigest()
        self.action_history.append({
            "action": action,
            "args_hash": args_hash,
            "result_hash": result_hash,
        })

    def to_summary(self) -> str:
        """
        Generate a COMPACT summary suitable for LLM context injection.

        Aggressively truncates to stay under ~1500 characters:
        - Only the last 5 observations
        - Only the last 5 errors
        - Source snippets capped at 100 chars each
        - Plan steps shown concisely

        Returns:
            A truncated string summary of the current task state.
        """
        parts: list = []

        # Header
        parts.append(f"[Task] {self.original_request[:120]}")
        parts.append(f"[Status] {self.status} | Step {self.current_step}/{len(self.plan)} | Iter {self.iteration_count}")

        # Plan (compact)
        if self.plan:
            plan_lines = []
            for i, step in enumerate(self.plan):
                marker = "→" if i == self.current_step else "·"
                step_text = step[:60] if isinstance(step, str) else str(step)[:60]
                plan_lines.append(f"  {marker} {i}: {step_text}")
            parts.append("[Plan]\n" + "\n".join(plan_lines))

        # Subagent results (just names and count)
        if self.subagent_results:
            agent_summary = ", ".join(
                f"{name}({len(results)})" for name, results in self.subagent_results.items()
            )
            parts.append(f"[Subagents] {agent_summary}")

        # Sources (truncated snippets)
        if self.sources_consulted:
            src_lines = []
            for src in self.sources_consulted[-5:]:  # Last 5 sources
                snippet = src.get("snippet", "")[:100]
                src_lines.append(f"  [{src['type']}] {src['ref']}: {snippet}")
            parts.append("[Sources]\n" + "\n".join(src_lines))

        # Files modified
        if self.files_modified:
            parts.append(f"[Files Modified] {', '.join(self.files_modified[-10:])}")

        # Observations (last 5, truncated)
        if self.observations:
            obs_lines = [f"  - {obs[:100]}" for obs in self.observations[-5:]]
            parts.append("[Observations]\n" + "\n".join(obs_lines))

        # Errors (last 5, truncated)
        if self.errors:
            err_lines = [f"  ❌ {err[:100]}" for err in self.errors[-5:]]
            parts.append("[Errors]\n" + "\n".join(err_lines))

        summary = "\n".join(parts)

        # Final hard truncation to ensure we stay under 1500 chars
        if len(summary) > 1500:
            summary = summary[:1497] + "..."

        return summary



# --- Confirmation ---
print("=" * 60)
print("✅ TaskState definido correctamente")
print("   Propiedades: original_request, status, plan, current_step,")
print("   subagent_results, sources_consulted, files_modified,")
print("   observations, errors, iteration_count, action_history")
print("   Métodos: add_source, add_observation, mark_file_modified,")
print("   add_error, record_action, to_summary")
print("=" * 60)


### 💾 Memoria persistente del proyecto

In [ ]:
# =============================================================================
# Cell 05 — Memory: ProjectMemory
# =============================================================================
# Persistent project memory backed by JSON files. Stores architectural
# knowledge, important files, dependencies, useful commands, conventions,
# decisions, bugs, and session summaries across agent sessions.
# get_context_summary() produces a compact string (< 800 chars) suitable
# for injection into LLM system prompts.
# =============================================================================

import json
import os


class ProjectMemory:
    """
    Persistent memory store for a project.

    Remembers architectural decisions, important files, dependencies,
    useful commands, coding conventions, bugs found, and session summaries.
    Data is serialized to/from JSON on disk.
    """

    # Default empty memory structure
    _DEFAULT_DATA = {
        "architecture": "",
        "important_files": {},    # {path: description}
        "dependencies": [],       # [dependency strings]
        "commands": {},           # {name: command_string}
        "conventions": [],        # [convention strings]
        "decisions": [],          # [decision strings]
        "bugs": [],               # [bug description strings]
        "session_summaries": [],  # [summary strings]
    }

    def __init__(self, project_name: str, base_dir: str = "./data/memory"):
        """
        Initialize project memory.

        Args:
            project_name: Identifier for the project (used as filename).
            base_dir: Directory where memory JSON files are stored.
        """
        self.project_name: str = project_name
        self.base_dir: str = base_dir
        # Deep copy the default data structure
        self.data: dict = json.loads(json.dumps(self._DEFAULT_DATA))

    @property
    def _file_path(self) -> str:
        """Full path to the memory JSON file."""
        return os.path.join(self.base_dir, f"{self.project_name}.json")

    def load(self):
        """
        Load memory from JSON file if it exists.

        Merges loaded data with defaults to ensure all keys are present
        even if the file was saved with an older schema.
        """
        try:
            if os.path.exists(self._file_path):
                with open(self._file_path, "r", encoding="utf-8") as f:
                    loaded = json.load(f)
                # Merge with defaults (preserves new keys added to schema)
                for key, default_value in self._DEFAULT_DATA.items():
                    if key not in loaded:
                        loaded[key] = (
                            json.loads(json.dumps(default_value))
                            if isinstance(default_value, (dict, list))
                            else default_value
                        )
                self.data = loaded
        except Exception as e:
            print(f"⚠️  Error al cargar memoria del proyecto: {e}")

    def save(self):
        """
        Save memory to JSON file, creating directories as needed.
        """
        try:
            os.makedirs(self.base_dir, exist_ok=True)
            with open(self._file_path, "w", encoding="utf-8") as f:
                json.dump(self.data, f, indent=2, ensure_ascii=False)
        except Exception as e:
            print(f"⚠️  Error al guardar memoria del proyecto: {e}")

    def update(self, section: str, data):
        """
        Update a section of memory by merging new data.

        Behavior depends on the section's data type:
        - str: replaces the value
        - dict: merges keys (update)
        - list: appends items (avoids duplicates for simple values)

        Args:
            section: The memory section key to update.
            data: The new data to merge into the section.
        """
        if section not in self.data:
            print(f"⚠️  Sección desconocida: '{section}'")
            return

        current = self.data[section]

        if isinstance(current, str):
            # For strings, just replace
            self.data[section] = str(data)

        elif isinstance(current, dict) and isinstance(data, dict):
            # For dicts, merge
            current.update(data)

        elif isinstance(current, list):
            # For lists, append new items
            if isinstance(data, list):
                for item in data:
                    if item not in current:
                        current.append(item)
            else:
                if data not in current:
                    current.append(data)

        else:
            # Fallback: just replace
            self.data[section] = data

    def get_context_summary(self) -> str:
        """
        Generate a compact summary (< 800 chars) for LLM system prompts.

        Includes architecture overview, key files, useful commands,
        and recent decisions. Designed to give the agent quick context
        about the project without consuming too many tokens.

        Returns:
            A compact string summary of project memory.
        """
        parts: list = []

        # Architecture (truncated)
        arch = self.data.get("architecture", "")
        if arch:
            parts.append(f"[Arch] {arch[:150]}")

        # Important files (top 5)
        files = self.data.get("important_files", {})
        if files:
            file_entries = list(files.items())[:5]
            file_lines = [f"  {path}: {desc[:40]}" for path, desc in file_entries]
            parts.append("[Key Files]\n" + "\n".join(file_lines))

        # Commands (top 4)
        commands = self.data.get("commands", {})
        if commands:
            cmd_entries = list(commands.items())[:4]
            cmd_lines = [f"  {name}: {cmd[:50]}" for name, cmd in cmd_entries]
            parts.append("[Commands]\n" + "\n".join(cmd_lines))

        # Recent decisions (last 3)
        decisions = self.data.get("decisions", [])
        if decisions:
            dec_lines = [f"  - {d[:60]}" for d in decisions[-3:]]
            parts.append("[Decisions]\n" + "\n".join(dec_lines))

        # Conventions (top 3)
        conventions = self.data.get("conventions", [])
        if conventions:
            conv_lines = [f"  - {c[:50]}" for c in conventions[:3]]
            parts.append("[Conventions]\n" + "\n".join(conv_lines))

        # Dependencies count
        deps = self.data.get("dependencies", [])
        if deps:
            parts.append(f"[Deps] {len(deps)} dependencies")

        summary = "\n".join(parts)

        # Hard truncation to stay under 800 chars
        if len(summary) > 800:
            summary = summary[:797] + "..."

        return summary if summary else "[No project memory available]"


# --- Confirmation ---
print("=" * 60)
print("✅ ProjectMemory definido correctamente")
print("   Secciones: architecture, important_files, dependencies,")
print("   commands, conventions, decisions, bugs, session_summaries")
print("   Métodos: load, save, update, get_context_summary")
print("   Persistencia: JSON en ./data/memory/<project_name>.json")
print("=" * 60)


### 🔍 Motor RAG (Retrieval-Augmented Generation)

Sistema de búsqueda semántica sobre documentación técnica de Kotlin y Spring Boot.
- **Chunking**: Fragmentos de 500 caracteres con 50 de overlap
- **Embeddings**: OpenAI `text-embedding-3-small`
- **Vector Store**: ChromaDB (persistente en disco)


In [ ]:
# ── Cell 06: RAG Engine (ChromaDB) ──────────────────────────────────────────────
# Uses globals from prior cells: client, EMBEDDING_MODEL

import chromadb
import hashlib

rag_collection = None


def init_rag(persist_dir: str = "./data/chroma"):
    """Initialize ChromaDB persistent client and collection."""
    global rag_collection
    try:
        chroma_client = chromadb.PersistentClient(path=persist_dir)
        rag_collection = chroma_client.get_or_create_collection(
            name="kotlin_springboot_docs",
            metadata={"hnsw:space": "cosine"},
        )
        print(f"✅ RAG inicializado — colección: 'kotlin_springboot_docs', "
              f"documentos existentes: {rag_collection.count()}")
    except Exception as e:
        print(f"❌ Error inicializando RAG: {e}")
        raise


def chunk_document(
    text: str,
    source_name: str,
    chunk_size: int = 500,
    overlap: int = 50,
) -> list:
    """Split *text* into overlapping character-based chunks.

    Returns list of {"text": str, "metadata": {"source": str, "chunk_index": int}}.
    """
    if not text or not text.strip():
        return []

    chunks: list = []
    start = 0
    index = 0

    while start < len(text):
        end = start + chunk_size
        chunk_text = text[start:end]

        # Skip empty trailing chunks
        if chunk_text.strip():
            chunks.append({
                "text": chunk_text,
                "metadata": {"source": source_name, "chunk_index": index},
            })
            index += 1

        # Advance by (chunk_size - overlap), but at least 1 char to avoid infinite loop
        step = max(chunk_size - overlap, 1)
        start += step

    return chunks


def embed_and_store(chunks: list, collection):
    """Generate embeddings via OpenAI and store in ChromaDB (batches of 100)."""
    if not chunks:
        return

    batch_size = 100
    for i in range(0, len(chunks), batch_size):
        batch = chunks[i : i + batch_size]
        texts = [c["text"] for c in batch]

        try:
            response = client.embeddings.create(
                model=EMBEDDING_MODEL,
                input=texts,
            )
            embeddings = [item.embedding for item in response.data]
        except Exception as e:
            print(f"❌ Error generando embeddings (batch {i // batch_size}): {e}")
            continue

        ids = []
        documents = []
        metadatas = []
        for c in batch:
            uid = hashlib.md5(
                f"{c['metadata']['source']}_{c['metadata']['chunk_index']}".encode()
            ).hexdigest()
            ids.append(uid)
            documents.append(c["text"])
            metadatas.append(c["metadata"])

        try:
            collection.upsert(
                ids=ids,
                documents=documents,
                embeddings=embeddings,
                metadatas=metadatas,
            )
        except Exception as e:
            print(f"❌ Error almacenando en ChromaDB (batch {i // batch_size}): {e}")


def rag_search(query: str, n_results: int = 5) -> str:
    """Search the RAG collection. Returns a formatted numbered list with source attribution."""
    global rag_collection
    if rag_collection is None or rag_collection.count() == 0:
        return "No se encontraron resultados relevantes."

    try:
        # Embed the query
        query_response = client.embeddings.create(
            model=EMBEDDING_MODEL,
            input=[query],
        )
        query_embedding = query_response.data[0].embedding

        results = rag_collection.query(
            query_embeddings=[query_embedding],
            n_results=min(n_results, rag_collection.count()),
        )

        if not results["documents"] or not results["documents"][0]:
            return "No se encontraron resultados relevantes."

        docs = results["documents"][0]
        metadatas = results["metadatas"][0]
        distances = results["distances"][0]

        formatted_lines: list = []
        for idx, (doc, meta, dist) in enumerate(zip(docs, metadatas, distances), 1):
            # Cosine distance → similarity (ChromaDB returns distance, lower = better)
            relevance = max(0.0, 1.0 - dist)
            source = meta.get("source", "desconocido")
            snippet = doc[:200].replace("\n", " ")
            formatted_lines.append(
                f"{idx}. [Fuente: {source}] (relevancia: {relevance:.2f})\n   {snippet}..."
            )

        return "\n".join(formatted_lines)

    except Exception as e:
        return f"Error en búsqueda RAG: {e}"


def ingest_document(text: str, source_name: str, collection=None):
    """Convenience: chunk + embed + store. Uses global rag_collection if *collection* is None."""
    global rag_collection
    target = collection if collection is not None else rag_collection
    if target is None:
        print("❌ No hay colección RAG disponible. Ejecutá init_rag() primero.")
        return

    chunks = chunk_document(text, source_name)
    if not chunks:
        print(f"⚠️  Documento '{source_name}' vacío o sin contenido útil, se omitió.")
        return

    embed_and_store(chunks, target)
    print(f"📄 Documento '{source_name}' ingestado — {len(chunks)} chunks almacenados.")


# ── Initialization ──────────────────────────────────────────────────────────────
init_rag()
print(f"🔍 RAG Engine listo — colección contiene {rag_collection.count()} documentos.")


### 🔧 Sistema de Herramientas (Tools)

Registro de herramientas con validación de políticas de seguridad.
Cada herramienta se registra con su esquema OpenAI, función de ejecución y tipo de permiso.


In [ ]:
# ── Cell 07: Tool Registry & Implementations ────────────────────────────────────
# Uses globals from prior cells: config, check_permission, TAVILY_API_KEY,
#                                 rag_search (cell 06), tracer, WORKSPACE

import os
import subprocess
import json
import time
import hashlib

from tavily import TavilyClient


def get_tools_for_agent(tool_names: list) -> tuple:
    """Return (schemas_list, fn_map) filtered to *tool_names*."""
    schemas: list = []
    fn_map: dict = {}
    for name in tool_names:
        entry = TOOL_REGISTRY.get(name)
        if entry:
            schemas.append(entry["schema"])
            fn_map[name] = entry["fn"]
    return schemas, fn_map


# ── Tool Implementations ────────────────────────────────────────────────────────

def tool_read_file(path: str) -> str:
    """Read and return the contents of a file."""
    try:
        allowed, reason = check_permission("read", path, config)
        if not allowed:
            return f"❌ Permiso denegado para leer '{path}': {reason}"
        with open(path, "r", encoding="utf-8", errors="replace") as f:
            return f.read()
    except Exception as e:
        return f"❌ Error leyendo '{path}': {e}"


def tool_write_file(path: str, content: str) -> str:
    """Write content to a file, creating parent directories if needed."""
    try:
        allowed, reason = check_permission("write", path, config)
        if not allowed:
            return f"❌ Permiso denegado para escribir '{path}': {reason}"
        os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
        with open(path, "w", encoding="utf-8") as f:
            f.write(content)
        return f"✅ Archivo escrito: {path} ({len(content)} caracteres)"
    except Exception as e:
        return f"❌ Error escribiendo '{path}': {e}"


def tool_list_files(directory: str) -> str:
    """List the contents of a directory."""
    try:
        allowed, reason = check_permission("read", directory, config)
        if not allowed:
            return f"❌ Permiso denegado para listar '{directory}': {reason}"
        if not os.path.isdir(directory):
            return f"❌ '{directory}' no es un directorio válido."
        entries = sorted(os.listdir(directory))
        result_lines: list = []
        for entry in entries:
            full = os.path.join(directory, entry)
            kind = "📁" if os.path.isdir(full) else "📄"
            result_lines.append(f"  {kind} {entry}")
        return f"Contenido de '{directory}' ({len(entries)} elementos):\n" + "\n".join(result_lines)
    except Exception as e:
        return f"❌ Error listando '{directory}': {e}"


def tool_run_command(command: str) -> str:
    """Run a shell command in the workspace directory and return stdout+stderr."""
    try:
        allowed, reason = check_permission("command", command, config)
        if not allowed:
            return f"❌ Permiso denegado para ejecutar '{command}': {reason}"

        # Run from workspace directory
        workspace_dir = config.get("workspace", WORKSPACE) if config else "."
        result = subprocess.run(
            command,
            shell=True,
            capture_output=True,
            text=True,
            timeout=120,
            cwd=workspace_dir or ".",
        )
        output = ""
        if result.stdout:
            output += result.stdout
        if result.stderr:
            output += ("\n--- STDERR ---\n" + result.stderr) if output else result.stderr
        if not output.strip():
            output = "(sin salida)"
        return f"[exit code: {result.returncode}]\n{output}"

    except subprocess.TimeoutExpired:
        return f"❌ Timeout: el comando excedió 120 segundos: '{command}'"
    except Exception as e:
        return f"❌ Error ejecutando comando: {e}"


def tool_web_search(query: str) -> str:
    """Search the web via Tavily and return formatted results."""
    try:
        tavily = TavilyClient(api_key=TAVILY_API_KEY)
        response = tavily.search(query=query, max_results=5)
        results = response.get("results", [])
        if not results:
            return "No se encontraron resultados web."
        formatted: list = []
        for i, r in enumerate(results, 1):
            title = r.get("title", "Sin título")
            url = r.get("url", "")
            snippet = r.get("content", "")[:300]
            formatted.append(f"{i}. **{title}**\n   URL: {url}\n   {snippet}")
        return "\n\n".join(formatted)
    except Exception as e:
        return f"❌ Error en búsqueda web: {e}"


def tool_rag_search(query: str, n_results: int = 5) -> str:
    """Search the RAG knowledge base."""
    try:
        return rag_search(query, n_results=n_results)
    except Exception as e:
        return f"❌ Error en búsqueda RAG: {e}"


# ── execute_tool_call ────────────────────────────────────────────────────────────

def execute_tool_call(tool_call, allowed_fns: dict, config: dict,
                      tracer=None, trace=None, state=None) -> str:
    """Execute a single tool call with policy check, logging, and state updates."""
    tool_name = tool_call.function.name
    start_time = time.time()

    try:
        args = json.loads(tool_call.function.arguments)
    except json.JSONDecodeError as e:
        return f"❌ Error parseando argumentos de '{tool_name}': {e}"

    # Check if tool is allowed for this agent
    if tool_name not in allowed_fns:
        return f"❌ Herramienta '{tool_name}' no permitida para este sub-agente."

    # Check permission via registry
    registry_entry = TOOL_REGISTRY.get(tool_name, {})
    perm_type = registry_entry.get("permission_type")
    if perm_type:
        # Determine target for permission check
        target = args.get("path") or args.get("directory") or args.get("command") or args.get("query", "")
        allowed, reason = check_permission(perm_type, target, config)
        if not allowed:
            return f"❌ Permiso denegado para '{tool_name}' sobre '{target}': {reason}"

    # Execute
    fn = allowed_fns[tool_name]
    
    # Supervisión Global (Human-in-the-loop)
    if config.get("supervision_mode") and tool_name in ["write_file", "run_command"]:
        action_desc = f"escribir en '{args.get('path')}'" if tool_name == "write_file" else f"ejecutar el comando '{args.get('command')}'"
        user_input = input(f"🔒 [Supervisión] El agente intenta {action_desc}.\n¿Aprobar acción? (s/n): ")
        if user_input.strip().lower() not in ("s", "si", "sí", "y", "yes"):
            return f"🚫 Acción rechazada por el usuario en modo supervisión."
    
    try:
        result = fn(**args)
    except TypeError as e:
        result = f"❌ Error en argumentos de '{tool_name}': {e}"
    except Exception as e:
        result = f"❌ Error ejecutando '{tool_name}': {e}"

    duration_ms = (time.time() - start_time) * 1000

    # Log to tracer
    if tracer and trace:
        try:
            tracer.log_tool_call(trace, tool_name, args, str(result)[:500], duration_ms)
            if tool_name == "rag_search":
                tracer.log_rag_retrieval(trace, args.get("query", ""), [{"content": str(result)[:1500]}])
        except Exception:
            pass  # Don't let tracing break the flow

    # Update state
    if state:
        try:
            args_str = json.dumps(args, ensure_ascii=False)
            state.record_action(tool_name, args_str, str(result)[:200])

            # Track sources for rag and web searches
            if tool_name == "rag_search":
                state.add_source("rag", args.get("query", ""), str(result)[:100])
            elif tool_name == "web_search":
                state.add_source("web", args.get("query", ""), str(result)[:100])
            elif tool_name == "write_file":
                state.mark_file_modified(args.get("path", ""))
        except Exception:
            pass

    return str(result)


# ── Static Tool Registry ───────────────────────────────────────────────────────────

TOOL_REGISTRY: dict = {
    "read_file": {
        "schema": {
            "type": "function",
            "function": {
                "name": "read_file",
                "description": "Lee el contenido de un archivo dado su path.",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "path": {"type": "string", "description": "Ruta absoluta al archivo a leer."},
                    },
                    "required": ["path"],
                },
            },
        },
        "fn": tool_read_file,
        "permission_type": "read",
    },
    "write_file": {
        "schema": {
            "type": "function",
            "function": {
                "name": "write_file",
                "description": "Escribe contenido en un archivo. Crea directorios intermedios si no existen.",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "path": {"type": "string", "description": "Ruta absoluta al archivo a escribir."},
                        "content": {"type": "string", "description": "Contenido a escribir en el archivo."},
                    },
                    "required": ["path", "content"],
                },
            },
        },
        "fn": tool_write_file,
        "permission_type": "write",
    },
    "list_files": {
        "schema": {
            "type": "function",
            "function": {
                "name": "list_files",
                "description": "Lista el contenido de un directorio.",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "directory": {"type": "string", "description": "Ruta absoluta al directorio a listar."},
                    },
                    "required": ["directory"],
                },
            },
        },
        "fn": tool_list_files,
        "permission_type": "read",
    },
    "run_command": {
        "schema": {
            "type": "function",
            "function": {
                "name": "run_command",
                "description": "Ejecuta un comando de shell en el directorio del workspace y devuelve stdout+stderr.",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "command": {"type": "string", "description": "Comando de shell a ejecutar."},
                    },
                    "required": ["command"],
                },
            },
        },
        "fn": tool_run_command,
        "permission_type": "command",
    },
    "web_search": {
        "schema": {
            "type": "function",
            "function": {
                "name": "web_search",
                "description": "Busca información en la web usando Tavily. Devuelve hasta 5 resultados.",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "query": {"type": "string", "description": "Consulta de búsqueda web."},
                    },
                    "required": ["query"],
                },
            },
        },
        "fn": tool_web_search,
        "permission_type": None,
    },
    "rag_search": {
        "schema": {
            "type": "function",
            "function": {
                "name": "rag_search",
                "description": "Busca en la base de conocimiento RAG (documentación Kotlin/Spring Boot indexada).",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "query": {"type": "string", "description": "Consulta de búsqueda en la base RAG."},
                        "n_results": {"type": "integer", "description": "Número de resultados (default 5)."},
                    },
                    "required": ["query"],
                },
            },
        },
        "fn": tool_rag_search,
        "permission_type": None,
    },
}

# ── Confirmation ─────────────────────────────────────────────────────────────────
print("🛠️  Tool Registry inicializado.")
print(f"   Herramientas registradas ({len(TOOL_REGISTRY)}):")
for name, entry in TOOL_REGISTRY.items():
    perm = entry["permission_type"] or "sin restricción"
    print(f"   • {name} — permiso: {perm}")


### 🧠 Gestión de contexto y detección de loops

- **ContextManager**: Resume conversaciones largas para no exceder la ventana de contexto
- **LoopDetector**: Detecta acciones repetidas sin progreso y fuerza cambio de estrategia


In [ ]:
# ── Cell 08: Context Management & Loop Detection ────────────────────────────────
# Uses globals from prior cells: client, MODEL

import hashlib


class ContextManager:
    """Manages conversation context length by estimating tokens and summarizing
    older messages when the context approaches its limit."""

    def __init__(self, max_tokens: int = 20000):
        self.max_tokens = max_tokens

    def estimate_tokens(self, messages: list) -> int:
        """Rough estimate: 1 token ≈ 4 characters."""
        total_chars = 0
        for msg in messages:
            content = msg.get("content") or ""
            total_chars += len(content)
        return total_chars // 4

    def should_summarize(self, messages: list) -> bool:
        """True if estimated tokens exceed 70 % of max_tokens."""
        return self.estimate_tokens(messages) > int(self.max_tokens * 0.7)

    def summarize_history(self, messages: list) -> list:
        """Summarize older messages via the LLM, keeping the system prompt and
        the last 6 messages intact.

        Returns a new messages list with the middle section replaced by a
        summary injected as a system message.
        """
        if len(messages) <= 7:
            # Nothing meaningful to summarize (system + ≤6 messages)
            return messages

        system_msg = messages[0]
        last_6 = messages[-6:]
        middle = messages[1:-6]

        if not middle:
            return messages

        # Build a summary request for the middle messages
        middle_text = "\n".join(
            f"[{m.get('role', '?')}]: {(m.get('content') or '')[:300]}"
            for m in middle
        )

        try:
            summary_response = client.chat.completions.create(
                model=MODEL,
                messages=[
                    {
                        "role": "system",
                        "content": (
                            "Summarize this conversation concisely, preserving "
                            "key decisions, file changes, and errors."
                        ),
                    },
                    {"role": "user", "content": middle_text},
                ],
                max_tokens=500,
            )
            summary_text = summary_response.choices[0].message.content or ""
        except Exception as e:
            summary_text = f"(Error al resumir: {e})"

        summary_msg = {
            "role": "system",
            "content": f"Previous conversation summary: {summary_text}",
        }

        return [system_msg, summary_msg] + last_6


class LoopDetector:
    """Detects when a subagent repeats the same action+args+result too many times."""

    def __init__(self, max_repeats: int = 3):
        self.max_repeats = max_repeats
        self.history: list = []  # list of (action, args_hash, result_hash)

    def record(self, action: str, args_hash: str, result_hash: str):
        """Record an action for loop detection."""
        self.history.append((action, args_hash, result_hash))

    def check(self) -> str | None:
        """Check for loops at the tail of the history.

        Returns a warning string if the last *max_repeats* entries are identical,
        otherwise returns None.
        """
        if len(self.history) < self.max_repeats:
            return None

        tail = self.history[-self.max_repeats:]
        if len(set(tail)) == 1:
            action, _, _ = tail[0]
            return (
                f"⚠️ Loop detectado: la acción '{action}' se repitió "
                f"{self.max_repeats} veces con los mismos argumentos y resultado. "
                f"Intentá un enfoque diferente."
            )
        return None

    def reset(self):
        """Clear history."""
        self.history.clear()


# ── Confirmation ─────────────────────────────────────────────────────────────────
print("🧠 ContextManager y LoopDetector listos.")


### 🤖 Motor de Subagentes

5 subagentes especializados, cada uno con su propio prompt, herramientas permitidas y ciclo de ejecución.


In [ ]:
# ── Cell 09: Subagent Engine ─────────────────────────────────────────────────────
# Uses globals from prior cells: client, MODEL, config, tracer,
#                                 get_tools_for_agent, execute_tool_call

import json
import time
import hashlib

# ── System Prompts (Kotlin / Spring Boot specialization) ─────────────────────────

EXPLORER_PROMPT = """You are the **Explorer** subagent for a Kotlin/Spring Boot coding assistant.

Your mission is to thoroughly explore and map the repository structure.

**Focus areas:**
- `build.gradle.kts` / `settings.gradle.kts` — dependencies, plugins, Kotlin version
- `application.yml` / `application.properties` — configuration, profiles
- Package structure under `src/main/kotlin/` and `src/test/kotlin/`
- Spring annotations: @RestController, @Service, @Repository, @Entity, @SpringBootApplication
- Entry point(s) — classes annotated with @SpringBootApplication
- Key files: controllers, services, repositories, models, DTOs

**Tools available:** list_files, read_file, run_command (for find/grep)

**Output:** A structured report with:
1. Architecture overview (layered? hexagonal?)
2. Entry points and main packages
3. Key files with brief descriptions
4. Dependencies (from build.gradle.kts)
5. Potential issues or observations
"""

RESEARCHER_PROMPT = """You are the **Researcher** subagent for a Kotlin/Spring Boot coding assistant.

**Important workflow:**
1. ALWAYS search the RAG knowledge base FIRST using `rag_search`.
2. Only use `web_search` as a FALLBACK when RAG has no relevant results.
3. Use `read_file` to examine specific code when needed.

**Always cite your source type** for every piece of information:
- `[RAG]` — from the indexed documentation
- `[WEB]` — from web search
- `[INFERENCE]` — your own reasoning based on context

**Focus on:**
- Kotlin official documentation: null safety, data classes, when expressions, extension functions, coroutines
- Spring Boot official docs: dependency injection, auto-configuration, annotations, testing
- Spring Data JPA: repository interfaces, query methods, custom queries
- Gradle Kotlin DSL

**Be concise.** Provide actionable findings, not lengthy explanations.
"""

IMPLEMENTER_PROMPT = """You are the **Implementer** subagent for a Kotlin/Spring Boot coding assistant.

**Write idiomatic Kotlin:**
- Use `data class` for DTOs and models
- Use null safety: `?.` (safe call), `?:` (Elvis), `let`, `also`; avoid `!!`
- Use constructor injection: `@Service class Foo(private val bar: Bar)`
- Use `when` expressions instead of long if-else chains
- Use extension functions where they improve readability

**Follow Spring Boot conventions:**
- `@RestController` + `@RequestMapping` for REST endpoints
- `@Service` for business logic
- `@Repository` for data access
- `ResponseEntity<T>` for HTTP responses with proper status codes
- `@Valid` for input validation with Jakarta Bean Validation

**Rules:**
1. Always use `read_file` to check the current state of a file BEFORE writing.
2. Make minimal, focused changes — don't rewrite entire files unless necessary.
3. Always explain WHAT you changed and WHY.
4. Create parent directories if needed.

**Tools available:** read_file, write_file, list_files
"""

TESTER_PROMPT = """You are the **Tester** subagent for a Kotlin/Spring Boot coding assistant.

**Your job is to RUN tests and REPORT results. Do NOT fix code.**

**How to run tests:**
- For Gradle projects: `cd {workspace} && ./gradlew test --no-daemon 2>&1`
- If gradlew is not executable: `cd {workspace} && gradle test --no-daemon 2>&1`

**Parse the output carefully:**
- Look for JUnit test results: PASSED / FAILED counts
- If tests FAIL, extract:
  - Exact test class and method that failed
  - Error message
  - Relevant stack trace (first 10 lines)
  - Expected vs. actual values if present

**Report format:**
1. Test command used
2. Overall result: ✅ ALL PASSED / ❌ FAILURES DETECTED
3. Summary: X passed, Y failed, Z skipped
4. For each failure:
   - Test: `ClassName.methodName`
   - Error: exact message
   - Stack trace excerpt

**DO NOT attempt to fix code. Only report findings.**

**Tools available:** run_command, read_file
"""

REVIEWER_PROMPT = """You are the **Reviewer** subagent for a Kotlin/Spring Boot coding assistant.

**Review code changes** by reading the modified files (current version) and comparing
with original versions provided in the conversation context.

**Check for:**
1. **Idiomatic Kotlin**: data classes, null safety (`?.`, `?:`), `when`, extension functions, no `!!`
2. **Spring conventions**: proper annotations, constructor injection, ResponseEntity usage
3. **Null safety**: no unsafe `!!` calls, proper Optional/nullable handling
4. **Proper annotations**: @Valid, @RequestBody, @PathVariable, correct HTTP method annotations
5. **Test coverage**: are there tests for new/changed functionality?
6. **Edge cases**: empty inputs, null values, concurrent access, error handling

**Output a structured review:**
```
## Code Review

### Verdict: APPROVE | REQUEST_CHANGES

### Summary
(1–2 sentence overview)

### Findings
- ✅ Good: ...
- ⚠️ Suggestion: ...
- ❌ Issue: ...

### Recommendations
(actionable list if REQUEST_CHANGES)
```

**Tools available:** run_command, read_file
"""

# ── Tool mapping per subagent ────────────────────────────────────────────────────

SUBAGENT_TOOLS: dict = {
    "explorer":    ["list_files", "read_file", "run_command"],
    "researcher":  ["rag_search", "web_search", "read_file"],
    "implementer": ["read_file", "write_file", "list_files"],
    "tester":      ["run_command", "read_file"],
    "reviewer":    ["run_command", "read_file"],
}

# Prompt lookup
_SUBAGENT_PROMPTS: dict = {
    "explorer":    EXPLORER_PROMPT,
    "researcher":  RESEARCHER_PROMPT,
    "implementer": IMPLEMENTER_PROMPT,
    "tester":      TESTER_PROMPT,
    "reviewer":    REVIEWER_PROMPT,
}


# ── Core subagent runner ─────────────────────────────────────────────────────────

def run_subagent(
    name: str,
    task: str,
    state,
    config: dict,
    tracer=None,
    trace=None,
    memory=None,
) -> str:
    """Run a subagent with its own message history and filtered tools.

    Manages loop detection, context summarization, tracing, and state updates.
    Returns the subagent's final text response.
    """
    max_iterations = config.get("max_iterations", 15) if config else 15

    # ── 1. Resolve prompt and tools ──────────────────────────────────────────
    system_prompt_text = _SUBAGENT_PROMPTS.get(name)
    if system_prompt_text is None:
        return f"❌ Sub-agente desconocido: '{name}'"

    tool_names = SUBAGENT_TOOLS.get(name, [])
    schemas, fn_map = get_tools_for_agent(tool_names)

    # ── 2. Build initial messages ────────────────────────────────────────────
    # Enrich system prompt with state summary and memory context
    enrichment_parts: list = [system_prompt_text.strip()]

    if state:
        try:
            summary = state.to_summary()
            if summary:
                enrichment_parts.append(f"\n--- Estado actual del task ---\n{summary}")
        except Exception:
            pass

    if memory:
        try:
            mem_ctx = memory.get_context_summary()
            if mem_ctx:
                enrichment_parts.append(f"\n--- Memoria del proyecto ---\n{mem_ctx}")
        except Exception:
            pass

    full_system = "\n".join(enrichment_parts)

    messages: list = [
        {"role": "system", "content": full_system},
        {"role": "user", "content": task},
    ]

    # ── 3. Initialize helpers ────────────────────────────────────────────────
    loop_detector = LoopDetector()
    context_manager = ContextManager()

    print(f"\n🤖 Sub-agente '{name}' iniciado...")
    print(f"   Tarea: {task[:120]}{'...' if len(task) > 120 else ''}")
    print(f"   Herramientas: {', '.join(tool_names)}")
    print(f"   Máx iteraciones: {max_iterations}")

    final_response = f"(El sub-agente '{name}' no generó respuesta)"

    # ── 4. LLM loop ─────────────────────────────────────────────────────────
    for iteration in range(1, max_iterations + 1):
        print(f"   ↻ Iteración {iteration}/{max_iterations}...", end="")

        try:
            call_start = time.time()
            response = client.chat.completions.create(
                model=MODEL,
                messages=messages,
                tools=schemas if schemas else None,
                tool_choice="auto" if schemas else None,
            )
            call_duration_ms = (time.time() - call_start) * 1000

            # Log LLM call to tracer
            if tracer and trace:
                try:
                    tracer.log_llm_call(trace, name, messages, response, call_duration_ms)
                except Exception:
                    pass

            assistant_msg = response.choices[0].message

        except Exception as e:
            error_text = f"❌ Error en llamada LLM (iteración {iteration}): {e}"
            print(f" {error_text}")
            if state:
                try:
                    state.add_error(error_text)
                except Exception:
                    pass
            final_response = error_text
            break

        # ── No tool calls → final response ──────────────────────────────────
        if not assistant_msg.tool_calls:
            final_response = assistant_msg.content or final_response
            print(f" ✅ Respuesta final recibida.")
            # Append to messages for completeness
            messages.append({"role": "assistant", "content": final_response})
            break

        # ── Process tool calls ───────────────────────────────────────────────
        print(f" 🔧 {len(assistant_msg.tool_calls)} tool call(s)")

        # Append the assistant message (with tool_calls) to history
        messages.append({
            "role": "assistant",
            "content": assistant_msg.content or "",
            "tool_calls": [
                {
                    "id": tc.id,
                    "type": "function",
                    "function": {
                        "name": tc.function.name,
                        "arguments": tc.function.arguments,
                    },
                }
                for tc in assistant_msg.tool_calls
            ],
        })

        for tc in assistant_msg.tool_calls:
            result = execute_tool_call(
                tc, fn_map, config,
                tracer=tracer, trace=trace, state=state,
            )

            # Append tool result to messages
            messages.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": str(result),
            })

            # Record for loop detection
            try:
                args_hash = hashlib.md5(tc.function.arguments.encode()).hexdigest()[:8]
                result_hash = hashlib.md5(str(result).encode()).hexdigest()[:8]
                loop_detector.record(tc.function.name, args_hash, result_hash)
            except Exception:
                pass

        # ── Loop detection ───────────────────────────────────────────────────
        loop_warning = loop_detector.check()
        if loop_warning:
            print(f"   {loop_warning}")
            messages.append({
                "role": "system",
                "content": loop_warning + " Cambiá tu estrategia o da tu respuesta final.",
            })

        # ── Context management ───────────────────────────────────────────────
        if context_manager.should_summarize(messages):
            print("   📝 Resumiendo historial de conversación...")
            try:
                messages = context_manager.summarize_history(messages)
            except Exception as e:
                print(f"   ⚠️ Error resumiendo: {e}")

        # ── Update state iteration count ─────────────────────────────────────
        if state:
            try:
                state.iteration_count += 1
            except Exception:
                pass

    else:
        # Max iterations reached without a final text response
        print(f"   ⚠️ Sub-agente '{name}' alcanzó el máximo de iteraciones ({max_iterations}).")
        final_response = (
            f"(El sub-agente '{name}' alcanzó el máximo de {max_iterations} "
            f"iteraciones sin respuesta final. Último contenido disponible.)"
        )
        # Try to get the last assistant content
        for msg in reversed(messages):
            if msg.get("role") == "assistant" and msg.get("content"):
                final_response = msg["content"]
                break

    # ── 5. Update state with results ─────────────────────────────────────────
    if state:
        try:
            if name not in state.subagent_results:
                state.subagent_results[name] = []
            state.subagent_results[name].append(final_response)
        except Exception:
            pass

    print(f"   🏁 Sub-agente '{name}' finalizado.\n")
    return final_response


# ── Confirmation ─────────────────────────────────────────────────────────────────
print("🤖 Motor de sub-agentes inicializado.")
print(f"   Sub-agentes disponibles: {', '.join(SUBAGENT_TOOLS.keys())}")
for sa_name, tools in SUBAGENT_TOOLS.items():
    print(f"   • {sa_name}: {', '.join(tools)}")


### 🎯 Agente Principal (Orquestador)

Coordina el flujo de trabajo con **routing no lineal**:
- Explorer → Researcher → Implementer ↔ Tester (loop de reintentos) → Reviewer
- Detecta cuando un subagente está bloqueado y cambia de estrategia
- Genera reportes finales con atribución de fuentes


In [ ]:
import sys
import os
import traceback
import json

def run_agent():
    """
    Main interactive loop. Handles:
    1. User input (chat, /plan on|off, /supervision on|off, exit)
    2. Creates TaskState per task
    3. Loads ProjectMemory
    4. Creates Langfuse trace
    5. Dispatches subagents with NON-LINEAR routing
    """
    plan_mode = False
    supervision_mode = False

    print("=================================================================")
    print("🤖 Coding Agent Avanzado (Kotlin/Spring Boot) iniciado.")
    print("   Comandos especiales: /plan on|off · /supervision on|off · exit")
    print("=================================================================\n")

    # Outer loop: read user input
    while True:
        try:
            user_input = input("\n👤 Usuario: ").strip()
        except EOFError:
            break

        if not user_input:
            continue

        lower_input = user_input.lower()
        if lower_input in ("exit", "quit"):
            print("👋 Terminando ejecución...")
            break

        if lower_input == "/plan on":
            plan_mode = True
            print("📋 Modo Plan ACTIVADO.")
            continue
        elif lower_input == "/plan off":
            plan_mode = False
            print("📋 Modo Plan DESACTIVADO.")
            continue
        elif lower_input == "/supervision on":
            supervision_mode = True
            print("🔒 Modo Supervisión ACTIVADO.")
            continue
        elif lower_input == "/supervision off":
            supervision_mode = False
            print("🔒 Modo Supervisión DESACTIVADO.")
            continue

        # Normal input processing
        print("\n⚙️ Iniciando nueva tarea...")
        
        # 2. Create TaskState
        state = TaskState(original_request=user_input)
        
        # 3. Load ProjectMemory
        project_name = os.path.basename(WORKSPACE.rstrip('/')) if WORKSPACE else "default_project"
        memory = ProjectMemory(project_name=project_name)
        memory.load()

        # 4. Start Langfuse trace
        trace = tracer.start_trace(name=f"Task: {user_input[:50]}...", user_id="user")

        try:
            # PLAN PHASE
            if plan_mode:
                print("📝 Generando plan de ejecución...")
                plan_messages = [
                    {"role": "system", "content": "You are a planning agent. Given a user request, output a step-by-step numbered plan to accomplish it. Do NOT output any tool calls, just text."},
                    {"role": "user", "content": user_input}
                ]
                response = client.chat.completions.create(
                    model=MODEL,
                    messages=plan_messages
                )
                plan_text = response.choices[0].message.content
                print(f"\n📋 PLAN PROPUESTO:\n{'-'*40}\n{plan_text}\n{'-'*40}")
                
                approved = False
                while True:
                    resp = input("¿Aprobás este plan? (y / n / modify): ").strip().lower()
                    if resp in ('y', 'yes', 's', 'si', 'sí'):
                        print("✅ Plan aprobado. Ejecutando...")
                        approved = True
                        break
                    elif resp in ('n', 'no'):
                        print("❌ Plan rechazado. Tarea cancelada.")
                        break
                    elif resp == 'modify':
                        feedback = input("Ingresá tus modificaciones: ").strip()
                        print("⏳ Re-generando plan (simplificado para este demo)...")
                        user_input = f"{user_input}. Note: {feedback}"
                        break
                    else:
                        print("⚠️ Respuesta inválida.")
                
                if not approved and resp in ('n', 'no'):
                    tracer.flush()
                    continue

            # ORCHESTRATION PHASE
            state.status = "exploring"
            print("\n🔍 Fase 1: Exploración del repositorio")
            config["supervision_mode"] = supervision_mode
            explorer_result = run_subagent("explorer", user_input, state, config, tracer, trace, memory)
            if "NOT_ENOUGH_INFO" in explorer_result or "BLOCKED" in explorer_result:
                print("⚠️ Explorer reporta que falta información o está bloqueado.")
            
            state.status = "researching"
            print("\n📚 Fase 2: Investigación (RAG y Web)")
            config["supervision_mode"] = supervision_mode
            researcher_result = run_subagent("researcher", user_input, state, config, tracer, trace, memory)

            state.status = "implementing"
            print("\n🛠️ Fase 3: Implementación y Testing")
            
            # Save original file contents of workspace files for reviewer
            original_files = {}
            if WORKSPACE and os.path.exists(WORKSPACE):
                import glob
                for filepath in glob.glob(os.path.join(WORKSPACE, '**/*.kt'), recursive=True) + glob.glob(os.path.join(WORKSPACE, '**/*.kts'), recursive=True):
                    try:
                        with open(filepath, 'r', encoding='utf-8') as file_obj:
                            original_files[filepath] = file_obj.read()
                    except Exception:
                        pass
            
            # Implementer / Tester loop
            max_retries = 3
            success = False
            for attempt in range(1, max_retries + 1):
                print(f"   ➤ Intento {attempt} de implementación...")
                if attempt > 1:
                    imp_task = f"{user_input}\n\nThe previous attempt failed testing. Please fix the code. Tester errors:\n{state.errors[-3:]}"
                imp_task = user_input
                    
                config["supervision_mode"] = supervision_mode  # Pasa el estado a las tools
                impl_result = run_subagent("implementer", imp_task, state, config, tracer, trace, memory)
                
                print("   ➤ Ejecutando tests...")
                config["supervision_mode"] = supervision_mode
                test_result = run_subagent("tester", "Run tests and verify the changes.", state, config, tracer, trace, memory)
                
                # Basic heuristic for success based on typical test outputs
                if ("PASSED" in test_result.upper() and "FAILED" not in test_result.upper()) or \
                   ("SUCCESS" in test_result.upper() and "ERROR" not in test_result.upper() and "EXCEPTION" not in test_result.upper()):
                     print("   ✅ Tests pasaron exitosamente.")
                     success = True
                     break
                else:
                    print(f"   ❌ Tests fallaron en el intento {attempt}.")
            
            if not success:
                print("⚠️ No se logró que los tests pasen después de los reintentos.")
            
            state.status = "reviewing"
            print("\n👀 Fase 4: Revisión de código")
            rev_task = f"{user_input}\n\nReview the changes. For context, these are some original files before changes (if modified):\n"
            for f in state.files_modified:
                if f in original_files:
                     rev_task += f"--- ORIGINAL {f} ---\n{original_files[f]}\n"
            
            config["supervision_mode"] = supervision_mode        
            reviewer_result = run_subagent("reviewer", rev_task, state, config, tracer, trace, memory)

            state.status = "completed"

            # GENERATE REPORT
            print("\n=================================================================")
            print("📝 REPORTE FINAL DE LA TAREA")
            print("=================================================================")
            print(f"Tarea: {user_input}")
            print(f"Archivos modificados: {', '.join(set(state.files_modified)) if state.files_modified else 'Ninguno'}")
            print("\nFuentes consultadas:")
            for src in state.sources_consulted:
                 print(f" - [{src['type']}] {src['ref']}")
            if state.errors:
                 print(f"\nErrores encontrados (últimos):")
                 for err in state.errors[-2:]:
                      print(f" - {err[:200]}...")
            print("\nResumen del Reviewer:")
            print(reviewer_result[:500] + ("..." if len(reviewer_result) > 500 else ""))
            print("=================================================================\n")

            # SAVE MEMORY
            print("💾 Guardando memoria del proyecto...")
            mem_summary_msgs = [
                {"role": "system", "content": "Extract architecture, important files, and commands from the following task state. Format as JSON with keys 'architecture', 'important_files' (dict), 'commands' (dict)."},
                {"role": "user", "content": state.to_summary()}
            ]
            try:
                mem_resp = client.chat.completions.create(model=MODEL, messages=mem_summary_msgs, response_format={"type": "json_object"})
                mem_data = json.loads(mem_resp.choices[0].message.content)
                if 'architecture' in mem_data: memory.update('architecture', mem_data['architecture'])
                if 'important_files' in mem_data: memory.update('important_files', mem_data['important_files'])
                if 'commands' in mem_data: memory.update('commands', mem_data['commands'])
                memory.save()
            except Exception as e:
                print(f"⚠️ Error actualizando memoria: {e}")

        except Exception as e:
            print(f"\n❌ Ocurrió un error inesperado durante la ejecución: {e}")
            traceback.print_exc()
        finally:
            print("🔌 Flushing observabilidad (Langfuse)...")
            tracer.flush()


### 📚 Ingesta de documentación para RAG

Carga documentación de Kotlin, Spring Boot, Gradle y patrones comunes en el vector store.


In [ ]:
# RAG Ingestion Cell

KOTLIN_NULL_SAFETY = """
# Kotlin Null Safety
Kotlin's type system is aimed at eliminating the danger of null references, also known as The Billion Dollar Mistake.
- Nullable types: String? can hold null, String cannot.
- Safe calls: `b?.length` returns length if b is not null, otherwise null.
- Elvis operator: `val l = b?.length ?: -1` provides a default value (-1) if the expression to the left is null.
- Not-null assertion operator: `b!!` converts any value to a non-null type and throws NullPointerException if the value is null. Use this sparingly!
- Safe casts: `a as? Int` returns null if the cast is unsuccessful.
- `let` function: `listWithNulls.forEach { it?.let { println(it) } }` executes the block only if `it` is not null.
"""

KOTLIN_DATA_CLASSES = """
# Kotlin Data Classes
Data classes are mainly used to store state.
`data class User(val name: String, val age: Int)`
Compiler automatically derives the following members from all properties declared in the primary constructor:
- equals()/hashCode() pair
- toString() of the form "User(name=John, age=42)"
- componentN() functions corresponding to the properties in their order of declaration
- copy() function: `val jack = user.copy(name = "Jack")`
To exclude a property from the generated implementations, declare it inside the class body instead of the primary constructor.
"""

KOTLIN_BASICS = """
# Kotlin Basics
- when expression: Replaces the switch operator.
  ```kotlin
  when (x) {
      1 -> print("x == 1")
      2 -> print("x == 2")
      else -> { print("x is neither 1 nor 2") }
  }
  ```
- Extension functions: Provides the ability to extend a class with new functionality without having to inherit from the class.
  `fun String.removeFirstLastChar(): String = this.substring(1, this.length - 1)`
- Sealed classes: Represent restricted class hierarchies. All direct subclasses of a sealed class are known at compile time.
- Companion objects: If you need to write a function that can be called without having a class instance but needs access to the internals of a class, you can write it as a member of a companion object declaration inside that class.
"""

SPRING_BOOT_REST = """
# Spring Boot REST Controllers
- `@RestController`: Combines @Controller and @ResponseBody.
- Mapping annotations: `@GetMapping("/path")`, `@PostMapping`, `@PutMapping`, `@DeleteMapping`.
- Extracting values:
  - `@PathVariable`: Extracts values from the URI path (e.g., `/api/users/{id}`).
  - `@RequestParam`: Extracts query parameters (e.g., `/api/users?name=john`).
  - `@RequestBody`: Binds the HTTP request body to a domain object.
- `ResponseEntity`: Represents the entire HTTP response (status code, headers, and body).
  - Return `ResponseEntity.ok(body)` for 200 OK.
  - Return `ResponseEntity.created(locationURI).body(body)` for 201 Created.
  - Return `ResponseEntity.notFound().build()` for 404 Not Found.
"""

SPRING_BOOT_DI = """
# Spring Boot Dependency Injection
- Stereotype annotations: `@Component`, `@Service`, `@Repository`, `@Controller`. These tell Spring to manage the class as a bean.
- Constructor Injection: The recommended way to inject dependencies, especially in Kotlin.
  ```kotlin
  @Service
  class UserService(private val userRepository: UserRepository) { ... }
  ```
- `@Autowired`: Optional for constructor injection if there's only one constructor. Used for field or setter injection (less preferred).
- `@Configuration` & `@Bean`: Used to explicitly declare beans in a configuration class.
"""

SPRING_DATA_JPA = """
# Spring Data JPA
- Repositories: Interface extending `CrudRepository<T, ID>` or `JpaRepository<T, ID>`.
- Query derivation: Create queries by defining method names (e.g., `findByLastName(lastName: String)`).
- Custom queries: Use `@Query("SELECT u FROM User u WHERE u.status = ?1")`.
- Entity: Mark domain classes with `@Entity`.
- IDs: Use `@Id` and `@GeneratedValue(strategy = GenerationType.IDENTITY)` for auto-incrementing primary keys.
"""

SPRING_BOOT_TESTING = """
# Spring Boot Testing
- `@SpringBootTest`: Loads the complete application context for integration testing.
- `@WebMvcTest(UserController::class)`: Test slice for testing only the web layer (controllers).
- `MockMvc`: Used to perform HTTP requests and assert responses in web tests.
- `@MockBean` (or `@MockkBean` if using MockK): Replaces a bean in the context with a mock.
- JUnit 5: Use `@Test`, `@BeforeEach`, `@AfterEach`.
- Assertions: `assertEquals(expected, actual)`, `assertNotNull(value)`, `assertThrows<Exception> { ... }`.
"""

GRADLE_KOTLIN_DSL = """
# Gradle Kotlin DSL
Structure of build.gradle.kts:
- `plugins { kotlin("jvm") version "1.9"; id("org.springframework.boot") version "3.2.0" }`
- `dependencies { implementation("org.springframework.boot:spring-boot-starter-web"); testImplementation("org.springframework.boot:spring-boot-starter-test") }`
- Dependencies configurations: `implementation` (required at compile and runtime), `testImplementation` (required only for tests).
"""

SPRING_VALIDATION = """
# Spring Validation
- Add dependency: `spring-boot-starter-validation`.
- Annotations on DTOs: `@NotBlank` (must not be null and must contain at least one non-whitespace character), `@NotNull`, `@Size(min=2, max=30)`, `@Pattern`.
- Trigger validation: Use `@Valid` on the `@RequestBody` parameter in the controller.
- Exception handling: If validation fails, a `MethodArgumentNotValidException` is thrown, which translates to a 400 Bad Request.
"""

print("📚 Iniciando ingesta de documentación en RAG...")

docs_to_ingest = [
    (KOTLIN_NULL_SAFETY, "Kotlin Docs - Null Safety"),
    (KOTLIN_DATA_CLASSES, "Kotlin Docs - Data Classes"),
    (KOTLIN_BASICS, "Kotlin Docs - Basics"),
    (SPRING_BOOT_REST, "Spring Docs - REST API"),
    (SPRING_BOOT_DI, "Spring Docs - Dependency Injection"),
    (SPRING_DATA_JPA, "Spring Docs - Data JPA"),
    (SPRING_BOOT_TESTING, "Spring Docs - Testing"),
    (GRADLE_KOTLIN_DSL, "Gradle Docs - Kotlin DSL"),
    (SPRING_VALIDATION, "Spring Docs - Validation")
]

for doc_text, source_name in docs_to_ingest:
    ingest_document(doc_text, source_name)

print(f"✅ Ingesta completada. Se indexaron {len(docs_to_ingest)} documentos principales.")
if rag_collection:
    print(f"📊 Total de chunks en ChromaDB: {rag_collection.count()}")


### 🏗️ Proyecto de demostración (Kotlin/Spring Boot)

Crea un proyecto Spring Boot con Kotlin que contiene bugs intencionales:
1. **Bug 1**: Uso inseguro de `!!` (non-null assertion) → `NullPointerException`
2. **Bug 2**: `ResponseEntity.ok()` en vez de `ResponseEntity.created()` (HTTP 201)
3. **Bug 3**: Sin validación de input (títulos vacíos permitidos)


In [ ]:
import os

# Create Demo Project Cell

DEMO_DIR = config.get("workspace", "./workspace/demo-spring-app")

directories = [
    "src/main/kotlin/com/demo/model",
    "src/main/kotlin/com/demo/repository",
    "src/main/kotlin/com/demo/service",
    "src/main/kotlin/com/demo/controller",
    "src/test/kotlin/com/demo/service"
]

for d in directories:
    os.makedirs(os.path.join(DEMO_DIR, d), exist_ok=True)

build_gradle_content = """plugins {
    kotlin("jvm") version "1.9.22"
    kotlin("plugin.spring") version "1.9.22"
    id("org.springframework.boot") version "3.2.2"
    id("io.spring.dependency-management") version "1.1.4"
}

group = "com.demo"
version = "0.0.1-SNAPSHOT"

java {
    sourceCompatibility = JavaVersion.VERSION_17
}

repositories {
    mavenCentral()
}

dependencies {
    implementation("org.springframework.boot:spring-boot-starter-web")
    implementation("org.springframework.boot:spring-boot-starter-validation")
    implementation("com.fasterxml.jackson.module:jackson-module-kotlin")
    implementation("org.jetbrains.kotlin:kotlin-reflect")
    testImplementation("org.springframework.boot:spring-boot-starter-test")
}

tasks.withType<Test> {
    useJUnitPlatform()
}
"""

settings_gradle_content = 'rootProject.name = "demo-spring-app"\n'

app_kt_content = """package com.demo

import org.springframework.boot.autoconfigure.SpringBootApplication
import org.springframework.boot.runApplication

@SpringBootApplication
class DemoApplication

fun main(args: Array<String>) {
    runApplication<DemoApplication>(*args)
}
"""

task_kt_content = """package com.demo.model

data class Task(
    val id: Long? = null,
    val title: String,
    val description: String = "",
    val completed: Boolean = false
)
"""

repo_kt_content = """package com.demo.repository

import com.demo.model.Task
import org.springframework.stereotype.Repository

@Repository
class TaskRepository {
    private val tasks = mutableMapOf<Long, Task>()
    private var nextId = 1L

    fun save(task: Task): Task {
        val saved = task.copy(id = task.id ?: nextId++)
        tasks[saved.id!!] = saved
        return saved
    }

    fun findById(id: Long): Task? = tasks[id]

    fun findAll(): List<Task> = tasks.values.toList()

    fun deleteById(id: Long) {
        tasks.remove(id)
    }
}
"""

service_kt_content = """package com.demo.service

import com.demo.model.Task
import com.demo.repository.TaskRepository
import org.springframework.stereotype.Service

@Service
class TaskService(private val repository: TaskRepository) {
    
    fun createTask(title: String, description: String = ""): Task {
        // BUG 3: No validation - allows empty title
        return repository.save(Task(title = title, description = description))
    }

    fun getTask(id: Long): Task {
        // BUG 1: Unsafe !! operator - will throw NPE if task not found
        return repository.findById(id)!!
    }

    fun getAllTasks(): List<Task> = repository.findAll()

    fun deleteTask(id: Long) = repository.deleteById(id)
}
"""

controller_kt_content = """package com.demo.controller

import com.demo.model.Task
import com.demo.service.TaskService
import org.springframework.http.ResponseEntity
import org.springframework.web.bind.annotation.*

@RestController
@RequestMapping("/api/tasks")
class TaskController(private val taskService: TaskService) {

    @GetMapping
    fun getAll(): ResponseEntity<List<Task>> = ResponseEntity.ok(taskService.getAllTasks())

    @GetMapping("/{id}")
    fun getById(@PathVariable id: Long): ResponseEntity<Task> = ResponseEntity.ok(taskService.getTask(id))

    @PostMapping
    fun create(@RequestBody request: CreateTaskRequest): ResponseEntity<Task> {
        val task = taskService.createTask(request.title, request.description)
        // BUG 2: Should return 201 Created, not 200 OK
        return ResponseEntity.ok(task)
    }
}

data class CreateTaskRequest(val title: String, val description: String = "")
"""

test_kt_content = """package com.demo.service

import com.demo.repository.TaskRepository
import org.junit.jupiter.api.Assertions.*
import org.junit.jupiter.api.BeforeEach
import org.junit.jupiter.api.Test
import org.junit.jupiter.api.assertThrows

class TaskServiceTest {
    private lateinit var repository: TaskRepository
    private lateinit var service: TaskService

    @BeforeEach
    fun setup() {
        repository = TaskRepository()
        service = TaskService(repository)
    }

    @Test
    fun `createTask should create a task with valid title`() {
        val task = service.createTask("Test Task", "Description")
        assertEquals("Test Task", task.title)
        assertNotNull(task.id)
    }

    @Test
    fun `getTask should return null-safe result for non-existent id`() {
        val exception = assertThrows<Exception> { service.getTask(999) }
        // We expect a specific domain exception or NullPointerException to be handled properly, 
        // not a raw Kotlin NullPointerException from the !! operator.
        assertFalse(exception is NullPointerException, "Should not throw raw NullPointerException")
    }

    @Test
    fun `createTask should reject empty title`() {
        // This test WILL FAIL due to BUG 3 (no validation)
        assertThrows<IllegalArgumentException> { service.createTask("") }
    }
}
"""

def write_demo_file(path, content):
    with open(os.path.join(DEMO_DIR, path), "w", encoding="utf-8") as f:
        f.write(content)

write_demo_file("build.gradle.kts", build_gradle_content)
write_demo_file("settings.gradle.kts", settings_gradle_content)
write_demo_file("src/main/kotlin/com/demo/DemoApplication.kt", app_kt_content)
write_demo_file("src/main/kotlin/com/demo/model/Task.kt", task_kt_content)
write_demo_file("src/main/kotlin/com/demo/repository/TaskRepository.kt", repo_kt_content)
write_demo_file("src/main/kotlin/com/demo/service/TaskService.kt", service_kt_content)
write_demo_file("src/main/kotlin/com/demo/controller/TaskController.kt", controller_kt_content)
write_demo_file("src/test/kotlin/com/demo/service/TaskServiceTest.kt", test_kt_content)

print(f"✅ Proyecto de demostración creado en: {DEMO_DIR}")
print("Contiene 3 bugs intencionales para que el agente los resuelva.")

# Change working directory so that commands run in the workspace context
os.chdir(DEMO_DIR)
print(f"CWD actualizado a: {os.getcwd()}")


### ▶️ Ejecutar Agente

Ejecutar esta celda para iniciar el agente interactivo.

**Comandos disponibles:**
- `/plan on` / `/plan off` — Activar/desactivar modo plan
- `/supervision on` / `/supervision off` — Activar/desactivar modo supervisión
- `exit` — Salir

**Tareas de prueba sugeridas:**
1. `"Explorá este proyecto, ejecutá los tests, encontrá los bugs, corregílos y verificá la corrección"`
2. `"Agregá validación de input al TaskService para que no se permitan títulos vacíos, y escribí tests"`
3. `"Optimizá el proyecto"` (tarea ambigua → el agente debería pedir clarificación)


In [ ]:
# Ejecutar Agente Cell

# Para iniciar el agente, simplemente llamá a run_agent().
# Podés escribir tu solicitud en español. Por ejemplo:
# "Explorá este proyecto, ejecutá los tests, encontrá los bugs, corregílos y verificá la corrección"

run_agent()
